In [ ]:
from spsa import SPSA
import llm
import dataset
import torch

In [ ]:

model_name = "distilbert-base-uncased"
dataset_name = "ag_news"
dataset_type = "MLM"

model_dict = llm.get_model(model_name)
model = model_dict["model"]

dataloader = dataset.get_dataset(dataset_name, dataset_type, model_dict["tokenizer"], 3)



In [ ]:
import random
print(model)
entries = sum([1 for _ in model.parameters()])

def add_to_param(model, pos, delta):
    for e, i in enumerate(model.parameters()):
        if e == pos:
            #print("WEIGHT NORM BEFORE", torch.norm(i), i)

            i += delta
            #print("WEIGHT NORM AFTER", torch.norm(i), i)
            return
def get_random_vec(model, pos, scaling=1e-6):
    for e, i in enumerate(model.parameters()):
        if e==pos:
            r = torch.rand(i.shape).to(device)
            return r*torch.mean(i)*1e-2

vec = torch.nn.utils.parameters_to_vector(model.parameters())
print(vec.shape)


lr = 1e-10
first = 0
scale = 1e-7
path="spsa_full.json"
lr_opt = [1e-6, 1e-7,1e-5]
scaling_opt = [1e-4,1e-5,1e-6]
for _ in range(10):
    for lr in lr_opt:
        for scale in scaling_opt:
            print("--------------------START---------------------------------")
            print("LR", lr, "Scale", scale)
            model = llm.get_model(model_name)["model"]
            epochs = 50
            to_save = {}
            to_save["lr"] = lr
            to_save["scaling"] = scale
            to_save["loss"] = []
            model.to(device)
            for epoch in range(epochs):
                total_loss = 0
                
                with torch.no_grad():
                    for batch in dataloader:
            
                        batch = {k: v.to(device) for k, v in batch.items()}
            
                        #print("------------------------------_START--------------------------")
                        vec = torch.nn.utils.parameters_to_vector(model.parameters())
            
                        delta = torch.rand(vec.shape).to(device)
            
                        vec_p = vec + delta*scale
                        vec_m = vec - delta*scale
                        #add_to_param(model,pos,delta)
                        torch.nn.utils.vector_to_parameters(vec_p, model.parameters())
            
                        loss_p = model(**batch).loss
            
            
                        torch.nn.utils.vector_to_parameters(vec_m, model.parameters())
            
                        loss_l = model(**batch).loss
                        
                        #print("loss_l", loss_l.item(), "loss_p", loss_p.item())
                        new = vec - delta*lr*((loss_p-loss_l)/(2*scale))
        
                        torch.nn.utils.vector_to_parameters(new, model.parameters())
            
                        
                        total_loss += model(**batch).loss.item()
                to_save["loss"].append(total_loss/len(dataloader))
                if epoch % 2 == 0:
                    print(first, f"Epoch {epoch+1}, Loss: {total_loss / len(dataloader):.4f}")
                if math.isnan(total_loss) or total_loss/len(dataloader) > 20:
                    break
            try:
                with open(path, "r") as f:
                    data = json.load(f)
            except (FileNotFoundError, json.JSONDecodeError):
                data = []
            data.append(to_save)
            with open(path, "w") as f:
                json.dump(data, f,indent=2)
            print("-----------------------------END---------------------")